# SlicerLive × IDC: view a segmentation from the Imaging Data Commons

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pieper/SlicerLive/blob/main/notebooks/SlicerLive_IDC_demo.ipynb)

This notebook uses [`idc-index`](https://github.com/ImagingDataCommons/idc-index) to find a DICOM **segmentation** in the NCI [Imaging Data Commons](https://imaging.datacommons.cancer.gov/) together with the **source image series it was drawn on**, then renders it in 3D + MPR with the [**SlicerLive**](https://github.com/pieper/SlicerLive) web viewer embedded right in the output cell.

Nothing is downloaded to this runtime — the viewer streams the DICOM directly from IDC's public AWS buckets and renders on your machine's GPU (WebGL).


## 1. Install idc-index

SlicerLive renders in the browser, so this notebook only needs `idc-index` to *find* data.

In [ ]:
!pip install -q idc-index

## 2. Find viewable segmentation cases

Each DICOM SEG references the exact image series it was drawn on (`segmented_SeriesInstanceUID` in `seg_index`). We join that to the main index to get the CRDC series UUID + AWS bucket for both the segmentation and its source CT/MR/PET — this is what the viewer needs.

In [ ]:
import urllib.parse
import pandas as pd
from idc_index import IDCClient

client = IDCClient()
client.fetch_index("seg_index")          # one row per DICOM SEG, incl. the source series it references
seg_index, main = client.seg_index, client.index

# (a) the SEG's own series row, and (b) the SOURCE image series it references
_seg = seg_index.merge(
    main[["SeriesInstanceUID", "crdc_series_uuid", "aws_bucket", "collection_id"]].rename(columns={
        "SeriesInstanceUID": "seg_suid", "crdc_series_uuid": "seg_crdc",
        "aws_bucket": "seg_bucket", "collection_id": "collection"}),
    left_on="SeriesInstanceUID", right_on="seg_suid", how="inner")
_src = main[["SeriesInstanceUID", "crdc_series_uuid", "aws_bucket", "Modality", "series_size_MB"]].rename(columns={
    "SeriesInstanceUID": "src_suid", "crdc_series_uuid": "src_crdc",
    "aws_bucket": "src_bucket", "Modality": "modality", "series_size_MB": "src_mb"})
cases = _seg.merge(_src, left_on="segmented_SeriesInstanceUID", right_on="src_suid", how="inner")

# Keep CT / MR / PET sources small enough to view comfortably in-browser
cases = cases[cases["modality"].isin(["CT", "MR", "PT"]) & (cases["src_mb"] <= 200)].reset_index(drop=True)
# ISPY1/ISPY2 breast MR (ispy1, ispy2, acrin_6698) render garbled from IDC pixel data — skip them
cases = cases[~cases["collection"].isin({"ispy1", "ispy2", "acrin_6698", "breast_mri_nact_pilot", "qiba_ct_1c", "lung_fused_ct_pathology"})].reset_index(drop=True)
print(f"{len(cases):,} viewable segmentation cases across {cases['collection'].nunique()} collections")
cases[["collection", "modality", "total_segments", "AlgorithmName", "src_mb"]].head()

## 3. Render one in 3D

`view(case)` builds a SlicerLive URL (`?ct=&seg=&mod=&ctb=&segb=`) and embeds the viewer. Re-run the cell for a new random case, or pass a specific row.

In [ ]:
from IPython.display import IFrame, HTML, display

VIEWER = "https://pieper.github.io/live/viewer.html"

def viewer_url(case):
    q = {"ct": case.src_crdc, "seg": case.seg_crdc, "mod": case.modality,
         "ctb": case.src_bucket, "segb": case.seg_bucket}
    return VIEWER + "?" + urllib.parse.urlencode(q)

def view(case, height=640):
    url = viewer_url(case)
    alg = case.AlgorithmName if pd.notna(case.AlgorithmName) else "segmentation"
    display(HTML(
        f'<div style="font:14px system-ui;margin:8px 0">'
        f'<b>{case.collection}</b> &middot; {case.modality} &middot; {int(case.total_segments)} segments '
        f'&middot; {alg} &nbsp; (<a href="{url}" target="_blank">open in a new tab</a>)</div>'))
    display(IFrame(url, width="100%", height=height))

view(cases.sample(1).iloc[0])

**Interacting with the viewer** (inside the cell above):

- **Drag** a slice or use the **wheel** to scroll through it; **right-drag** zoom, **middle-drag** pan.
- **Shift + move** over a slice to jump the other two views to that physical location.
- **Double-click** any panel to maximize it; double-click again to restore the 4-up.
- Toggle **Volume rendering** (top-right), and click **Details** for the collection summary + links to TCIA and the IDC viewer.
- Drag in the 3D view to rotate — the pivot is the segmentation's centroid.

## 4. Filter

Narrow by modality, collection, anatomy, or algorithm and view a match.

In [ ]:
# e.g. brain MR segmentations, or pick your own filter:
subset = cases[cases.modality == "MR"]
# subset = cases[cases.collection == "qin_prostate_repeatability"]
# subset = cases[cases.AlgorithmName.fillna("").str.contains("TotalSegmentator")]
print(f"{len(subset):,} matching")
view(subset.sample(1).iloc[0])

---
Data: NCI Imaging Data Commons (public). Viewer: [SlicerLive](https://github.com/pieper/SlicerLive) (vtk.js + dcmjs, client-side). The segmentation surfaces are generated in-browser with marching cubes; the source image renders as a thin-slab volume MPR + 3D volume rendering.